In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [4]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

115

In [5]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [9]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [28]:
from dotenv import load_dotenv
load_dotenv()
from google import genai
from google.genai import types
gemini_client = genai.Client()  


In [ ]:
import json
user_prompt = json.dumps(doc)
prompt = f"""{data_gen_instructions}\n\nFAQ record:\n{user_prompt}"""

In [36]:
response = gemini_client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Questions,
    ),
)

In [37]:
result = Questions.model_validate_json(response.text)
result.questions

['Is it too late to jump into the course if I just found out about it?',
 'Can I still sign up for the classes even though the course has already started?',
 'Will I be eligible for a certificate if I join the program this late?',
 'Do I need to worry about project deadlines if I decide to enroll now?',
 'Is there still a chance to get certified if I start working on the assignments today?']

In [41]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [42]:
from evaluation_utils import llm_structured

In [44]:
result, usage = llm_structured(
    gemini_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to jump into the course if I just found out about it?', 'Can I register for this now even though the sessions have already kicked off?', "Do I need to do anything specific if I'm joining late but still want to get a certificate?", 'Are there any penalties for signing up after the start date if I plan on submitting a final project?', 'Is there a deadline I need to watch for my project submission if I start late?']


In [45]:
usage

GenerateContentResponseUsageMetadata(
  candidates_token_count=117,
  prompt_token_count=264,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=264
    ),
  ],
  total_token_count=381
)

In [46]:
from evaluation_utils import calc_price

In [47]:
calc_price(usage)

{'input_cost': 0.00019800000000000002,
 'output_cost': 0.0005265,
 'total_cost': 0.0007245}

In [38]:
records = [
    {"question": question, "document": doc["id"]}
    for question in result.questions
]
records

[{'question': 'Is it too late to jump into the course if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': 'Can I still sign up for the classes even though the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'Will I be eligible for a certificate if I join the program this late?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to worry about project deadlines if I decide to enroll now?',
  'document': '74eb249bbf'},
 {'question': 'Is there still a chance to get certified if I start working on the assignments today?',
  'document': '74eb249bbf'}]

In [39]:
import pandas as pd

In [40]:
pd.DataFrame(records)

,question,document
0,Is it too late to jump into the course if I ju...,74eb249bbf
1,Can I still sign up for the classes even thoug...,74eb249bbf
2,Will I be eligible for a certificate if I join...,74eb249bbf
3,Do I need to worry about project deadlines if ...,74eb249bbf
4,Is there still a chance to get certified if I ...,74eb249bbf


In [48]:
from evaluation_utils import llm_structured_retry

In [49]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        gemini_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [50]:
generate_ground_truth(doc)

([{'question': 'Is it too late to jump into this course now?',
   'document': '74eb249bbf'},
  {'question': 'Can I catch up and finish the program if I start today?',
   'document': '74eb249bbf'},
  {'question': 'Will I be able to get a certificate if I join the class this late?',
   'document': '74eb249bbf'},
  {'question': 'What are the rules for getting certified if I missed the initial start date?',
   'document': '74eb249bbf'},
  {'question': 'Do I need to worry about deadlines to qualify for the certificate if I just enrolled?',
   'document': '74eb249bbf'}],
 GenerateContentResponseUsageMetadata(
   candidates_token_count=99,
   prompt_token_count=264,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=264
     ),
   ],
   total_token_count=363
 ))

In [51]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [52]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)
    # this here is implementing the parallize and i believe this is the reason its failing retry it again after a while when the limit resets. I will try to implement a retry mechanism in the map_progress function to handle this issue. or simply use the normal call method used aboved.

  0%|          | 0/115 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.1-flash-lite\nPlease retry in 31.163553256s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-

In [57]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

NameError: name 'results' is not defined

In [ ]:
ground_truth[10]

In [ ]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

In [ ]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)

In [ ]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

In [56]:
len(df_ground_truth)

NameError: name 'df_ground_truth' is not defined